# High-level developer example: scVI on Dataset 2 (mBDRC)

This short example shows the same method-agnostic flow on mBDRC. scRareBench loads dataset metadata (including the evaluation batch and registered rare-scenario taxonomy), the user runs scVI, and `benchmark_latent()` evaluates only the resulting latent.

> **Validation note:** this notebook was tested on Google Colab runtime 2026.07. An optional, fully commented compatibility-check cell is included for diagnostics only. Exact NumPy/PyTorch/JAX version matching is **not required** to run the notebook.


In [ ]:
# OPTIONAL: validate this runtime against the environment used for release testing.
# This check is informational only and is NOT required to run scRareBench or this notebook.
# Leave this cell unchanged to skip the check. Uncomment the lines below if you want to compare
# your current environment with the Google Colab 2026.07 runtime used during validation.
# A different compatible runtime may still work correctly.
#
# import sys
# from importlib import metadata as _runtime_metadata
#
# # This requirement belongs to this scVI/MrVI example, not to scRareBench itself.
# if sys.version_info < (3, 12):
#     print("Warning: scvi-tools==1.4.3 requires Python 3.12+.")
#
# _EXPECTED_COLAB_ANCHORS = {
#     "numpy": "2.0.2",
#     "torch": "2.11.0",
#     "jax": "0.7.2",
# }
# _runtime_mismatches = []
# for _package, _expected in _EXPECTED_COLAB_ANCHORS.items():
#     try:
#         _observed = _runtime_metadata.version(_package)
#     except _runtime_metadata.PackageNotFoundError:
#         _observed = "not installed"
#     if _observed != _expected:
#         _runtime_mismatches.append(
#             f"{_package}: validated {_expected}, current {_observed}"
#         )
#
# if _runtime_mismatches:
#     print("Runtime differs from the Colab 2026.07 validation environment:")
#     for _item in _runtime_mismatches:
#         print(" -", _item)
# else:
#     print("Runtime matches the documented Colab 2026.07 validation anchors.")


In [ ]:
import subprocess
import sys

URL = "git+https://github.com/amirhossein-alishahi/scRareBench_.git@v0.10.4"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", URL])

# The scVI dependency belongs to this example, not to scRareBench.
# setup_runtime preserves ABI-sensitive packages already present in this runtime.
from scrarebench.runtime import setup_runtime
setup_runtime(
    extra_requirements=("scvi-tools==1.4.3",),
    extra_imports=("scvi",),
    quiet=False,
)


In [ ]:
from scrarebench import load_dataset, dataset_info
adata = load_dataset(2)
info = dataset_info(adata)
print(adata)
print(info)

## User-owned method section: scVI

This block can be replaced by any integration method. scRareBench does not require a method adapter or registry entry.


In [ ]:
import numpy as np, pandas as pd, scanpy as sc, scipy.sparse as sp, scvi
BATCH_KEY = info["batch_key"]
if "counts" not in adata.layers:
    values = adata.X.data if sp.issparse(adata.X) else np.asarray(adata.X).ravel()
    sample = np.asarray(values[:min(len(values),100000)], dtype=float)
    if sample.size and (np.nanmin(sample) < 0 or not np.allclose(sample, np.rint(sample), atol=1e-6)):
        raise ValueError("Verify the raw-count source before scVI training.")
    adata.layers["counts"] = adata.X.copy()
method_adata = adata.copy()
sc.pp.highly_variable_genes(method_adata, layer="counts", flavor="seurat_v3", n_top_genes=min(4000, method_adata.n_vars), span=0.3, subset=False, check_values=True)
method_adata = method_adata[:, method_adata.var["highly_variable"].fillna(False).to_numpy()].copy()
scvi.settings.seed = 42
scvi.model.SCVI.setup_anndata(method_adata, layer="counts", batch_key=BATCH_KEY)
model = scvi.model.SCVI(method_adata, n_hidden=128, n_latent=30, n_layers=2, dropout_rate=0.10, dispersion="gene-batch", gene_likelihood="nb")
model.train(max_epochs=200, train_size=0.90, validation_size=0.10, batch_size=256, early_stopping=True, early_stopping_patience=20, accelerator="auto", devices="auto")
latent = model.get_latent_representation()
assert np.array_equal(method_adata.obs_names.astype(str), adata.obs_names.astype(str))

In [ ]:
from scrarebench import benchmark_latent
latent_df = pd.DataFrame(latent, index=adata.obs_names.astype(str))
result = benchmark_latent(adata, latent_df, method="scVI", config={"random_state": 42})

In [ ]:
from IPython.display import display
display(result.summary().round(5))
print("Interactive report:", result.interactive_report_path)
print("PDF:", result.pdf_path)
print("Bundle:", result.bundle_path)